<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/query_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Required Libraries
!pip install -qU langgraph langchain langchain-community langchain-ollama chromadb pydantic sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1

In [2]:
# Setup Environment (Ollama & Vector DB)
# 1. Unzip the vector database we created in Phase 0
import os, shutil, zipfile
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. Delete previous folders
for folder in ["methodology_db", "vector_db"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# 2. extract vector_db.zip to vector_db folder
with zipfile.ZipFile("vector_db.zip", 'r') as zip_ref:
    zip_ref.extractall("vector_db")

# 3. check if it has data
db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=OllamaEmbeddings(model="nomic-embed-text"))

/tmp/ipykernel_22617/1397332499.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=OllamaEmbeddings(model="nomic-embed-text"))


In [3]:
# 2. Install and run Ollama in the background
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time
print("[*] Starting Ollama server...")
subprocess.Popen(["nohup", "ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# We will use llama3 as our local reasoning agent
print("[*] Pulling llama3 model (This will take a few minutes)...")
!ollama pull llama3
print("[*] Pulling nomic-embed-text model...")
!ollama pull nomic-embed-text
print("[*] Setup complete!")

[*] Starting Ollama server...
[*] Pulling llama3 model (This will take a few minutes)...

[*] Pulling nomic-embed-text model...

[*] Setup complete!


In [5]:
# LangGraph Node 1 Implementation
import json
from typing import List, TypedDict
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate


In [6]:
# 1. DEFINE THE LANGGRAPH STATE
# This memory object will be passed between all nodes.
class UnifiedGraphState(TypedDict):
    # --- Input ---
    research_question: str

    # --- Node 1: NLP Extraction Outputs (SNOMED Metadata) ---
    primary_condition: str
    related_conditions: List[str]
    observations: List[str]
    concept_type: str
    snomed_top_hierarchy: str
    search_terms: List[str]
    explicit_exclusions: List[str]
    relevant_guidelines: List[str]
    suggested_validation_sources: List[str]
    ambiguity_notes: str

    # --- Node 1: Methodology RAG Output (Crucial for Node 4 Justification) ---
    qof_rules_text: str

    # --- Prepared for Future Nodes (Node 2 & 3) ---
    candidate_codes: List[dict]
    final_codes: List[dict]

In [7]:
# 2. LLM STRUCTURED OUTPUT SCHEMA (PYDANTIC)
# Forces the LLM to return exactly this JSON structure
class QueryExtraction(BaseModel):
    primary_condition: str = Field(description="The main disease or condition (e.g., 'Type 2 Diabetes')")
    related_conditions: List[str] = Field(description="Other mentioned conditions", default_factory=list)
    observations: List[str] = Field(description="Tests, measurements, or biomarkers (e.g., 'HbA1c', 'BMI', 'Blood Pressure')", default_factory=list)
    concept_type: str = Field(description="SNOMED concept type (e.g., 'disease', 'finding', 'procedure')")
    snomed_top_hierarchy: str = Field(description="Top level SNOMED hierarchy category (e.g., 'Clinical finding')")
    search_terms: List[str] = Field(description="List of optimized search synonyms for the TRUD/SNOMED database")
    explicit_exclusions: List[str] = Field(description="Conditions or keywords that must be explicitly excluded", default_factory=list)
    relevant_guidelines: List[str] = Field(description="Short names of inferred guidelines (e.g., ['NG28', 'QOF 2025/26'])", default_factory=list)
    suggested_validation_sources: List[str] = Field(description="Suggested sources for validation (e.g., ['OpenCodelists'])", default_factory=list)
    ambiguity_notes: str = Field(description="Any missing or ambiguous info in the user query", default="")


In [8]:
# 3. INITIALIZE MODELS & VECTOR DATABASE
print("[*] Initializing local models...")
# Connect to the offline vector database created in Phase 0
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=embeddings)

# Initialize the local LLM with structured output mapping
llm = ChatOllama(model="llama3", temperature=0)
structured_llm = llm.with_structured_output(QueryExtraction)

[*] Initializing local models...


In [9]:
# 4. NODE 1 CORE FUNCTION: QUERY UNDERSTANDING & RAG
def node_1_query_understanding(state: UnifiedGraphState) -> UnifiedGraphState:
    """
    Node 1 parses the user's research question into structured clinical entities,
    and retrieves the corresponding official QOF/NICE rules from the local Vector DB.
    """
    print("\n--- [NODE 1] QUERY UNDERSTANDING & METHODOLOGY RAG STARTED ---")

    query = state["research_question"]
    print(f"[*] Processing Query: '{query}'")

    # STEP A: LLM Decomposition (Entity Extraction)
    print("[*] Decomposing query via structured LLM...")
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an expert clinical data analyst working for the NHS. Your job is to extract medical entities from research questions to query the SNOMED CT UK Clinical Edition database. Be highly precise."),
        ("user", "Research Question: {question}")
    ])

    # Pipe the prompt to the structured LLM
    chain = prompt_template | structured_llm
    extracted_data = chain.invoke({"question": query})

    print(f"[*] Primary Condition: {extracted_data.primary_condition}")
    print(f"[*] Synonyms/Search Terms: {extracted_data.search_terms}")

    # STEP B: Methodology RAG (Retrieve Rules from ChromaDB)
    print("[*] Querying local Vector DB for clinical guidelines and QOF rules...")
    qof_rules_payload = ""
    found_guidelines = set() # Store metadata sources dynamically

    # We search the vector DB for both the primary and related conditions
    search_queries = [extracted_data.primary_condition] + extracted_data.related_conditions

    for sq in search_queries:
        print(f"    -> Retrieving official rules for: {sq}")
        # Fetch the top 2 most relevant guideline chunks per condition
        docs = vector_db.similarity_search(f"{sq} QOF rules inclusion exclusion criteria", k=2)

        for doc in docs:
            # Capture metadata (Source PDF and Page Number) for later justification
            source = doc.metadata.get('source', 'Unknown Document')
            page = doc.metadata.get('page', 'Unknown Page')

            # Extract just the filename (e.g., 'NG28_Type2_Diabetes')
            filename = source.split('/')[-1].replace('.pdf', '')
            found_guidelines.add(filename)

            qof_rules_payload += f"--- Source: {filename} (Page {page}) ---\n"
            qof_rules_payload += doc.page_content + "\n\n"

    print(f"[*] Retrieved {len(qof_rules_payload)} characters of raw guideline text.")

    # STEP C: Update and Return the Unified State
    print("[*] Updating UnifiedGraphState...")

    # Update the state dictionary with the newly acquired data
    state.update({
        "primary_condition": extracted_data.primary_condition,
        "related_conditions": extracted_data.related_conditions,
        "observations": extracted_data.observations,
        "concept_type": extracted_data.concept_type,
        "snomed_top_hierarchy": extracted_data.snomed_top_hierarchy,
        "search_terms": extracted_data.search_terms,
        "explicit_exclusions": extracted_data.explicit_exclusions,
        "relevant_guidelines": list(found_guidelines),
        "suggested_validation_sources": extracted_data.suggested_validation_sources,
        "ambiguity_notes": extracted_data.ambiguity_notes,
        "qof_rules_text": qof_rules_payload,
        # Ensure future lists exist so downstream nodes don't throw errors
        "candidate_codes": state.get("candidate_codes", []),
        "final_codes": state.get("final_codes", [])
    })

    print("--- [NODE 1] COMPLETE ---")
    return state

In [11]:
# 5. Test Node 1
if __name__ == "__main__":
    # Create an initial mock state to simulate an incoming analyst request
    initial_state: UnifiedGraphState = {
        "research_question": "Find active SNOMED codes for adult patients with Type 2 Diabetes who have a BMI of 30 or above. Exclude suspected cases.",
        "primary_condition": "",
        "related_conditions": [],
        "observations": [],
        "concept_type": "",
        "snomed_top_hierarchy": "",
        "search_terms": [],
        "explicit_exclusions": [],
        "relevant_guidelines": [],
        "suggested_validation_sources": [],
        "ambiguity_notes": "",
        "qof_rules_text": "",
        "candidate_codes": [],
        "final_codes": []
    }

    # Execute Node 1
    result_state = node_1_query_understanding(initial_state)

    # Print a summary to verify the state update was successful
    print("\n=== FINAL STATE SUMMARY ===")
    print(f"Primary Condition: {result_state['primary_condition']}")
    print(f"Observations: {result_state['observations']}")
    print(f"Exclusions: {result_state['explicit_exclusions']}")
    print(f"Guidelines Cited (Metadata): {result_state['relevant_guidelines']}")
    print(f"\nExtracted QOF Rules Payload (first 300 chars):\n{result_state['qof_rules_text'][:300]}...")


--- [NODE 1] QUERY UNDERSTANDING & METHODOLOGY RAG STARTED ---
[*] Processing Query: 'Find active SNOMED codes for adult patients with Type 2 Diabetes who have a BMI of 30 or above. Exclude suspected cases.'
[*] Decomposing query via structured LLM...
[*] Primary Condition: Type 2 Diabetes
[*] Synonyms/Search Terms: ['Type 2 diabetes', 'Diabetes mellitus, type II']
[*] Querying local Vector DB for clinical guidelines and QOF rules...
    -> Retrieving official rules for: Type 2 Diabetes
[*] Retrieved 1625 characters of raw guideline text.
[*] Updating UnifiedGraphState...
--- [NODE 1] COMPLETE ---

=== FINAL STATE SUMMARY ===
Primary Condition: Type 2 Diabetes
Observations: []
Exclusions: ['Suspected case of diabetes']
Guidelines Cited (Metadata): ['NG28_type_2_diabetes', 'qof_combined']

Extracted QOF Rules Payload (first 300 chars):
--- Source: qof_combined (Page 18) ---
MH003, MH006, MH007, M011 and MH012 Reporting and verification 
i. See indicator wording for requirement criteria